<a href="https://colab.research.google.com/github/nawrotlab/Nawrot_CNS_Course_solutions/blob/main/PythonProgramming_Day4_solutions.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Introduction to Iterative Maps and Attractor Networks - Day 4 - Solutions

In Days 1 to 3 you worked with variables, loops, functions, NumPy arrays, plotting, and simple data handling. In this notebook we build on those tools and use them for **generative models**: simple update rules that create a pattern over many repeated steps.

The goal here is not heavy mathematics. Instead, we use programming intuition:

- a system has a **state**
- a rule updates that state step by step
- after many updates, interesting structures can emerge

We will study two examples:

1. **Barnsley fern**: a stochastic iterative map that generates a plant-like fractal.
2. **Hopfield network (2D)**: a small attractor network that can store and recover a few patterns.

Later in the course you will meet continuous-time models, ordinary differential equations (ODEs), and Euler solvers. Here we stay in the simpler setting of **discrete-time updates**.

## Difference equations and iterative maps

A **difference equation** updates a quantity in discrete steps. In compact notation:

$$
\mathrm{state}_{t+1} = F(\mathrm{state}_t)
$$

This means: start with some initial state, apply the update rule once, then again, then again.

Some systems settle into a stable value or a stable pattern. Such a stable outcome is called an **attractor**.

For this notebook, the most important idea is:

- **loops** implement repeated updates
- **arrays** store the current state
- **plots** help us see how the state evolves

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from numpy.random import default_rng

rng = default_rng(42)
plt.rcParams["figure.figsize"] = (6, 6)

## A very small warm-up example

Before we move to the two main exercises, here is a simple one-dimensional iterative map:

$$
x_{t+1} = 0.85\,x_t + 0.15
$$

If you start from almost any value, repeated updates move the system toward a stable value near 1. This is the simplest kind of attractor.

In [ ]:
x = 3.0
trajectory = [x]

for _ in range(25):
    x = 0.85 * x + 0.15
    trajectory.append(x)

plt.plot(trajectory, marker="o")
plt.xlabel("iteration")
plt.ylabel("x")
plt.title("Simple iterative map approaching an attractor")
plt.grid(True)
plt.show()

## Exercise 1: Barnsley fern

The Barnsley fern is a classic example of an iterative map that creates a complex visual pattern from a few simple rules.

At every step, the current point $(x_t, y_t)$ is transformed into a new point $(x_{t+1}, y_{t+1})$ using one of several affine maps:

$$
\begin{pmatrix}x_{t+1} \\ y_{t+1}\end{pmatrix}
= A_k
\begin{pmatrix}x_t \\ y_t\end{pmatrix}
+ b_k
$$

The index $k$ is chosen randomly, but not with equal probability. Some maps are used much more often than others.

Even though each update is simple, the repeated random application of the rules generates a fern-like shape.

**Tasks**

1. Store the four linear maps and their offset vectors in NumPy arrays.
2. Store the corresponding probabilities.
3. Start from the point `(0, 0)`.
4. Iterate the map many times, for example `50_000` or `100_000` steps.
5. At each step:
   - choose one transformation using the probabilities
   - update the current point
   - store the new point
6. Plot all generated points with a very small marker size.
7. Remove the axes and choose a green color map or a dark green point color.

**Hints**

- `rng.choice(...)` lets you draw one transformation index using the probability vector `p`.
- `A[k] @ state + b[k]` applies the chosen affine map to the current point.
- `np.zeros((n_points, 2))` is a convenient way to store all generated points.
- `plt.scatter(...)` with a very small marker size works well for the final visualization.

**Further reading**

- Barnsley fern on Wikipedia: https://en.wikipedia.org/wiki/Barnsley_fern
- Barnsley fern generator with different variants: https://www.chradams.co.uk/fern/maker.html

**Questions**

- Which part of the code introduces randomness?
- Why does the point cloud not fill the entire plane?
- How does the picture change if you use fewer points?

In [ ]:
# Barnsley fern parameters
A = np.array([
    [[0.00, 0.00], [0.00, 0.16]],
    [[0.85, 0.04], [-0.04, 0.85]],
    [[0.20, -0.26], [0.23, 0.22]],
    [[-0.15, 0.28], [0.26, 0.24]],
])

b = np.array([
    [0.0, 0.0],
    [0.0, 1.6],
    [0.0, 1.6],
    [0.0, 0.44],
])

p = np.array([0.01, 0.85, 0.07, 0.07])
n_points = 50_000

### Solution

In [ ]:
points = np.zeros((n_points, 2))
state = np.zeros(2)

for i in range(n_points):
    k = rng.choice(len(p), p=p)
    state = A[k] @ state + b[k]
    points[i] = state

plt.scatter(points[:, 0], points[:, 1], s=0.05, color="darkgreen")
plt.axis("off")
plt.show()


## Exercise 2: A small 2D Hopfield network

A Hopfield network is a simple model of **associative memory**. It stores patterns in its weight matrix and can often recover a stored pattern from a noisy or incomplete version.

Here we use a small 2D version. Think of an `8 x 8` image made of black and white pixels. Each pixel is one unit in the network and has state `+1` or `-1`.

The basic idea is:

- you define a few patterns that should be remembered
- you compute weights from those patterns
- you start from a noisy version of one pattern
- repeated updates move the network toward a stored pattern

This is a simple example of an **attractor network**.

We keep the mathematics light. The most important operations are matrix multiplication, outer products, and repeated updates in a loop.

**Tasks**

1. Define at least three binary patterns with shape `(8, 8)` and values `-1` and `1`.
   Examples: letters, a square, a cross, a smiley, or any other simple symbols.
2. Flatten each pattern into a one-dimensional vector of length `64`.
3. Train a Hopfield weight matrix with the Hebbian rule:

$$
W = \frac{1}{N} \sum_\mu p^{(\mu)} {p^{(\mu)}}^T
$$

   where `N` is the number of units and each stored pattern is a vector `p`.
4. Set the diagonal of `W` to zero so that units do not directly reinforce themselves.
5. Create a noisy version of one stored pattern by flipping a small number of pixels.
6. Implement an **asynchronous update** rule:
   - choose one unit at a time
   - compute how strongly the other units support its current state
   - update that unit so that it follows the sign of this support
7. Repeat many asynchronous updates and see whether the original pattern is recovered.
8. Plot the original pattern, the noisy pattern, and the recovered pattern side by side.

**Questions**

- How many patterns can your small network store before retrieval gets worse?
- What happens if the noisy input is too strongly corrupted?
- In what sense is the final stored pattern an attractor?

In [ ]:
def show_pattern(pattern, ax=None, title=""):
    """Display a pattern with values -1 and 1."""
    if ax is None:
        ax = plt.gca()
    ax.imshow(pattern, cmap="Greys", vmin=-1, vmax=1)
    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)


def corrupt_pattern(pattern, n_flips, rng):
    """Flip n_flips randomly chosen entries of a pattern."""
    noisy = pattern.copy().reshape(-1)
    flip_idx = rng.choice(noisy.size, size=n_flips, replace=False)
    noisy[flip_idx] *= -1
    return noisy.reshape(pattern.shape)

In [ ]:
pattern_1 = -np.ones((8, 8), dtype=int)
pattern_1[1:7, 1] = 1
pattern_1[1:7, 6] = 1
pattern_1[3:5, 1:7] = 1

pattern_2 = -np.ones((8, 8), dtype=int)
pattern_2[1:7, 1:3] = 1
pattern_2[1:3, 1:7] = 1
pattern_2[3:5, 1:6] = 1
pattern_2[5:7, 1:7] = 1

pattern_3 = -np.ones((8, 8), dtype=int)
pattern_3[1:7, 3:5] = 1
pattern_3[3:5, 1:7] = 1

patterns_2d = [pattern_1, pattern_2, pattern_3]

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
for ax, pattern, title in zip(axes, patterns_2d, ["H", "S", "cross"]):
    show_pattern(pattern, ax=ax, title=title)
plt.tight_layout()
plt.show()


In [ ]:
def train_hopfield(patterns):
    n_units = patterns[0].size
    W = np.zeros((n_units, n_units), dtype=float)
    for pattern in patterns:
        p = pattern.reshape(-1)
        W += np.outer(p, p)
    W /= n_units
    np.fill_diagonal(W, 0)
    return W


def update_async(state, W, rng, n_steps):
    state = state.copy()
    for _ in range(n_steps):
        unit = rng.integers(state.size)
        support = W[unit] @ state
        if support != 0:
            state[unit] = np.sign(support)
    return state


### Suggested workflow for the Hopfield exercise

If you get stuck, work in small steps:

1. First make sure your three patterns really look different.
2. Train the weight matrix and print its shape.
3. Corrupt one pattern by flipping only a few pixels.
4. Run a small number of updates and inspect intermediate results.
5. Only then increase the number of update steps.

In [ ]:
W = train_hopfield(patterns_2d)
original = patterns_2d[0]
noisy = corrupt_pattern(original, n_flips=10, rng=rng)
recovered = update_async(noisy.reshape(-1), W, rng=rng, n_steps=1000)
recovered = recovered.reshape(original.shape)

fig, axes = plt.subplots(1, 3, figsize=(9, 3))
show_pattern(original, ax=axes[0], title="original")
show_pattern(noisy, ax=axes[1], title="noisy")
show_pattern(recovered, ax=axes[2], title="recovered")
plt.tight_layout()
plt.show()


### Optional: animate the Hopfield dynamics

Once your Hopfield network can recover a pattern, it is useful to watch how the network state changes over time. The next cell stores intermediate states during asynchronous updates and turns them into an animation.

In [ ]:
def collect_async_history(initial_state, W, rng, n_steps, snapshot_every=4):
    """Store intermediate network states using your update_async function."""
    state = initial_state.copy()
    history = [state.copy()]

    for step in range(n_steps):
        state = update_async(state, W, rng=rng, n_steps=1)

        if (step + 1) % snapshot_every == 0:
            history.append(state.copy())

    return np.array(history)


def animate_hopfield_history(history, shape=(8, 8), interval=200):
    """Create a matplotlib animation from a sequence of Hopfield states."""
    fig, ax = plt.subplots(figsize=(4, 4))
    image = ax.imshow(history[0].reshape(shape), cmap="Greys", vmin=-1, vmax=1)
    ax.set_xticks([])
    ax.set_yticks([])

    def update(frame):
        image.set_data(history[frame].reshape(shape))
        ax.set_title(f"Hopfield update snapshot {frame}")
        return [image]

    anim = FuncAnimation(fig, update, frames=len(history), interval=interval, blit=True)
    plt.close(fig)
    return anim


# Example usage after you have defined W, original, and noisy:
# history = collect_async_history(noisy.reshape(-1), W, rng=rng, n_steps=400, snapshot_every=4)
# anim = animate_hopfield_history(history, shape=original.shape, interval=150)
# HTML(anim.to_jshtml())

## Wrap-up

These two models are very different, but they share one important idea: **repeated updates generate structure**.

- In the **Barnsley fern**, iteration generates a geometric object.
- In the **Hopfield network**, iteration drives the system toward a stored memory.

Both are examples of simple computational models that produce nontrivial behavior from compact rules.

Later, when you study ODEs and Euler solvers, you will meet a related idea in continuous time. For now, iterative maps are a good first step because they connect directly to loops, arrays, and visualization.